# Lesson 24 — Neural-Network-Embedded MINLP

## Learning objectives

1. Encode a trained feedforward NN as algebraic constraints in an MINLP.
2. Distinguish ReLU big-M, full-space, and reduced-space formulations.
3. Use `discopt.nn` to embed a Keras / ONNX model.
4. Apply NN-MINLP to surrogate-based optimization.

## 1. Why embed a NN?

You have a trained NN that approximates an expensive physics simulation. You want to *optimize* over its inputs subject to other (algebraic) constraints. Three approaches:

- **Black-box** (Bayesian / surrogate-based): cheap iterations, no certificate.
- **Embed as constraint**: solver sees the NN as algebraic equations; with MINLP, you can certify global optimality of the surrogate.
- **Differentiable layer** (Lesson 27): NN is a step inside a larger optimization-aware pipeline.

NN-MINLP took off after {cite:p}`Fischetti2018` showed ReLU networks reduce to MILP, and {cite:p}`Anderson2020` introduced strong formulations. {cite:p}`Ceccon2022` (OMLT) is the modern toolkit; `discopt.nn` follows the same patterns.

## 2. ReLU big-M

For a single ReLU $z = \max(0, w^\top x + b)$ with $x \in [\underline x, \overline x]$:

$$
\begin{aligned}
z &\ge w^\top x + b \\
z &\ge 0 \\
z &\le w^\top x + b - \underline a (1 - y) \\
z &\le \overline a y \\
y &\in \{0, 1\}
\end{aligned}
$$

with $\underline a, \overline a$ the lower/upper bounds on the pre-activation. Tight $\underline a, \overline a$ via interval propagation are critical {cite:p}`Anderson2020`.

## 3. Smooth activations (sigmoid, tanh)

For sigmoid $z = \sigma(a)$, the constraint is genuinely nonlinear. Two options:

- **Full-space**: keep $z = \sigma(a)$ as a constraint; solve as MINLP.
- **Reduced-space**: substitute the NN equations through, leaving a single nested expression. Smaller variable count but worse for spatial B&B.

{cite:p}`Fischetti2018,Ceccon2022`.

## 4. Bound propagation

Propagate input bounds layer by layer:

$$\underline z_l = w^+ \underline x + w^- \overline x + b, \quad \overline z_l = w^+ \overline x + w^- \underline x + b.$$

(Decomposing $w$ into positive and negative parts.) For deep networks, FBBT alone gives loose bounds; OBBT layer-by-layer is tighter. CROWN, IBP, and SymBP are NN-specific bound propagation methods {cite:p}`Anderson2020`.

In [ ]:
import numpy as np, discopt as do
import discopt.nn as dn

# Toy: a 2-input, 2-hidden, 1-output ReLU NN
np.random.seed(0)
W1 = np.random.normal(size=(2, 2)); b1 = np.zeros(2)
W2 = np.random.normal(size=(2, 1)); b2 = np.zeros(1)

net = dn.NetworkDefinition(input_dim=2, output_dim=1)
net.add_layer(dn.DenseLayer(W1, b1, activation=dn.Activation.RELU))
net.add_layer(dn.DenseLayer(W2, b2, activation=dn.Activation.LINEAR))

# Optimization: find (x1, x2) in [-1, 1]^2 maximizing the NN output
m = do.Model("nn_opt")
x = m.add_variables(2, lb=-1, ub=1, name="x")
y = m.add_variable(name="y")
dn.embed(m, net, inputs=x, outputs=[y], formulation="relu_bigm")
m.maximize(y)
r = m.solve(mode="global")
print("global opt input:", r.x, " output:", r.value(y))


## 5. Surrogate-based optimization

If your NN approximates an expensive simulation, then:

1. Sample inputs, evaluate simulation, train NN.
2. Optimize via NN-MINLP to find a candidate input.
3. Evaluate the simulation at that candidate.
4. Add to training set, retrain (or fine-tune); iterate.

This is **Bayesian-flavoured but with MINLP** instead of Gaussian processes. Useful when the simulation is structured (e.g., reactor design where physics gives you a known input domain).

## References
{cite:p}`Fischetti2018,Anderson2020,Ceccon2022,Wang2022`.